# GLD/GDX Pairs Trading: Cointegration & Kalman Filter Analysis

**Dataset:** GLD (SPDR Gold Trust ETF) and GDX (VanEck Gold Miners ETF)  
**Period:** 2010-01-01 to 2024-12-31  
**Source:** Yahoo Finance via `yfinance`

This notebook analyzes the cointegrated relationship between gold spot exposure (GLD) and gold mining equities (GDX). Both instruments are driven by gold prices, but GDX carries additional equity and operational risk premium — making the spread between them economically meaningful and potentially mean-reverting. The analysis proceeds from data cleaning through EDA, formal cointegration testing, static and dynamic (Kalman filter) spread construction, signal generation, and regime analysis.

In [ ]:
# Uncomment to install dependencies
# !pip install yfinance pandas numpy matplotlib seaborn statsmodels pykalman scipy

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from scipy import stats
import warnings
import os

warnings.filterwarnings('ignore')

# Plot styling
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['lines.linewidth'] = 1.4

np.random.seed(42)
print('Libraries loaded successfully.')

---
## Step 1: Data Collection & Cleaning

We pull daily OHLCV data for GLD and GDX from Yahoo Finance using `yfinance`. We keep only the **adjusted close** price, which accounts for corporate actions (splits, dividends). Both tickers are downloaded in a single call so they share the same date index from the start.

Cleaning decisions:
- **Forward fill** isolated NaN gaps (carries the last valid price forward — standard for ETF price data where gaps are typically one-day data vendor issues, not actual non-trading days)
- **Drop** any rows with NaNs that remain after forward fill (e.g., leading NaNs before either ETF had data)
- All decisions are documented in the missing value report below

In [ ]:
# --- Constants ---
TICKERS    = ['GLD', 'GDX']
START_DATE = '2010-01-01'
END_DATE   = '2024-12-31'

# --- Download ---
print(f"Downloading {TICKERS} from {START_DATE} to {END_DATE}...")
raw = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True, progress=False)

# auto_adjust=True means 'Close' already reflects adjusted prices
adj_close = raw['Close'][TICKERS].copy()
adj_close.columns = ['GLD_adj', 'GDX_adj']

print(f"\nRaw download shape : {adj_close.shape}")
print(f"Date range         : {adj_close.index[0].date()}  →  {adj_close.index[-1].date()}")
print(f"\nFirst 5 rows:")
display(adj_close.head())
print(f"\nLast 5 rows:")
display(adj_close.tail())

In [ ]:
def missing_value_report(df: pd.DataFrame) -> pd.DataFrame:
    """Return a DataFrame summarising missing values per column."""
    return pd.DataFrame({
        'Missing Count': df.isnull().sum(),
        'Missing %'    : (df.isnull().sum() / len(df) * 100).round(4),
        'Total Rows'   : len(df)
    })

print("=== Missing Value Report — Raw Download ===")
display(missing_value_report(adj_close))

In [ ]:
def clean_price_series(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """
    Clean a price DataFrame by forward-filling isolated NaN gaps
    then dropping any remaining NaNs (e.g., leading rows).

    Returns:
        cleaned DataFrame and a report dict documenting the transformations.
    """
    rows_before = len(df)
    nans_before = int(df.isnull().sum().sum())

    # Step 1: forward fill isolated gaps
    df_filled = df.ffill()
    nans_after_ffill = int(df_filled.isnull().sum().sum())
    filled_count = nans_before - nans_after_ffill

    # Step 2: drop any remaining NaNs (should only be leading rows)
    df_clean = df_filled.dropna()
    rows_after  = len(df_clean)
    rows_dropped = rows_before - rows_after

    report = {
        'rows_raw'       : rows_before,
        'nans_raw'       : nans_before,
        'nans_ffilled'   : filled_count,
        'rows_dropped'   : rows_dropped,
        'rows_final'     : rows_after,
        'nans_final'     : int(df_clean.isnull().sum().sum()),
    }
    return df_clean, report


adj_close_clean, clean_report = clean_price_series(adj_close)

print("=== Cleaning Summary ===")
for k, v in clean_report.items():
    print(f"  {k:<20}: {v}")

assert clean_report['nans_final'] == 0, "NaNs remain after cleaning — inspect data!"
print("\nAssertion passed: zero NaN values in cleaned DataFrame.")

In [ ]:
# Save checkpoints for reproducibility
os.makedirs('data', exist_ok=True)

adj_close.to_csv('data/gld_gdx_raw.csv')
adj_close_clean.to_csv('data/gld_gdx_clean.csv')

print("Saved:")
print("  data/gld_gdx_raw.csv   — raw download")
print("  data/gld_gdx_clean.csv — after forward-fill and dropna")
print(f"\nFinal clean DataFrame shape: {adj_close_clean.shape}")
print(f"Date range: {adj_close_clean.index[0].date()}  →  {adj_close_clean.index[-1].date()}")

---
## Step 2: Log-Price Transformation

We apply a natural log transform to both price series before any analysis. Three reasons:

1. **Variance stabilisation** — raw prices exhibit heteroskedasticity (variance grows with price level). Log prices have more homogeneous variance across time.
2. **Additivity of returns** — log returns `Δlog(P)` are additive across periods, making them the standard unit of analysis in financial econometrics.
3. **Cointegration prerequisite** — the Engle-Granger and Johansen tests assume log-linear relationships between price levels. Working in log space ensures the hedge ratio `β` in `log(GLD) = α + β·log(GDX) + ε` has a clean economic interpretation (elasticity).

We also compute log returns here since they are needed for ADF testing in Step 4.

In [ ]:
df = adj_close_clean.copy()

# Log prices
df['log_GLD'] = np.log(df['GLD_adj'])
df['log_GDX'] = np.log(df['GDX_adj'])

# Log returns (first difference of log prices = continuously compounded return)
df['ret_GLD'] = df['log_GLD'].diff()
df['ret_GDX'] = df['log_GDX'].diff()

# The first row has NaN returns — drop it from the returns columns only;
# we keep the full df intact and use .dropna() locally where needed.
returns = df[['ret_GLD', 'ret_GDX']].dropna()

print("Main DataFrame columns:", df.columns.tolist())
print(f"df shape             : {df.shape}")
print(f"returns shape        : {returns.shape}")
print("\nSample (last 5 rows):")
display(df.tail())

In [ ]:
# Quick descriptive stats on log prices and returns
print("=== Log Price Descriptive Statistics ===")
display(df[['log_GLD', 'log_GDX']].describe().round(4))

print("\n=== Log Return Descriptive Statistics ===")
display(returns.describe().round(6))

---
## Step 3: Exploratory Data Analysis — Visualizations

Before any formal modeling, we visualize the data to build intuition about the GLD/GDX relationship. Five plots are produced:

1. **Normalized prices** — co-movement and divergence over the full period
2. **Rolling 60-day correlation** — how stable the return correlation is through time
3. **Rolling 30-day volatility** — comparing risk levels and volatility clustering
4. **Log price ratio** — a naive first look at the spread before formal cointegration testing
5. **Return distributions** — fat tails relative to a normal distribution; relevant for signal threshold interpretation later

In [ ]:
# --- Plot 1: Normalized Price Series ---
# Divide each series by its first value so both start at 1.0.
# This puts GLD and GDX on the same scale despite different absolute price levels.

norm = df[['GLD_adj', 'GDX_adj']].div(df[['GLD_adj', 'GDX_adj']].iloc[0])

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(norm.index, norm['GLD_adj'], label='GLD (Gold ETF)', color='goldenrod', linewidth=1.5)
ax.plot(norm.index, norm['GDX_adj'], label='GDX (Gold Miners ETF)', color='steelblue', linewidth=1.5)

# Annotate key macro events
events = {
    '2011-09-06': ('Gold Peak\n(Sep 2011)', 0.18),
    '2020-03-16': ('COVID\nCrash',          -0.25),
    '2022-03-08': ('Rate Hike\nCycle',       0.18),
}
for date_str, (label, offset) in events.items():
    x = pd.Timestamp(date_str)
    y = norm['GLD_adj'].loc[norm.index >= x].iloc[0]
    ax.axvline(x, color='grey', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.text(x, y + offset, label, fontsize=8, ha='center', color='dimgrey')

ax.set_title('Normalized Price Series: GLD vs. GDX (2010–2024)', fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price (normalized to 1.0 at start)')
ax.legend()
plt.tight_layout()
plt.savefig('data/plot1_normalized_prices.png', dpi=150)
plt.show()

print("""
Caption: Both series track gold broadly but GDX dramatically underperforms GLD over the
full period — GLD roughly doubled while GDX ended near its 2010 starting value. This
divergence is driven by cost inflation, capex cycles, and equity risk in mining companies.
Despite the level divergence, short-term co-movement is strong. The spread between them
is the signal — and its long-run drift motivates using a proper estimated hedge ratio
rather than a naive 1:1 ratio.
""")

In [ ]:
# --- Plot 2: Rolling 60-Day Correlation ---
# Correlation computed on log returns (not price levels) — return correlation is the
# standard measure and avoids the spurious correlation of non-stationary price series.

roll_corr = returns['ret_GLD'].rolling(60).corr(returns['ret_GDX'])
mean_corr  = roll_corr.mean()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(roll_corr.index, roll_corr, color='mediumseagreen', linewidth=1.3,
        label='60-day rolling correlation')
ax.axhline(mean_corr, color='crimson', linestyle='--', linewidth=1.2,
           label=f'Full-sample mean: {mean_corr:.3f}')
ax.axhline(0, color='black', linewidth=0.7, alpha=0.3)
ax.fill_between(roll_corr.index, roll_corr, mean_corr,
                where=(roll_corr < mean_corr), alpha=0.15, color='crimson',
                label='Below-mean periods')

ax.set_title('Rolling 60-Day Return Correlation: GLD vs. GDX', fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Pearson Correlation')
ax.set_ylim(-0.1, 1.05)
ax.legend()
plt.tight_layout()
plt.savefig('data/plot2_rolling_correlation.png', dpi=150)
plt.show()

print(f"""
Caption: The rolling 60-day return correlation averages {mean_corr:.3f}, confirming a
strong and persistent co-movement relationship between GLD and GDX. Periods where
correlation dips sharply (highlighted in red) correspond to macro stress events where
the two instruments temporarily decouple — miners sell off harder during broad equity
drawdowns (COVID 2020) while physical gold acts as a safe haven. These correlation
breakdowns are potential regime shifts where the cointegration relationship comes under
stress, examined further in the regime analysis section.
""")

In [ ]:
# --- Plot 3: Rolling 30-Day Volatility ---
# Annualised by multiplying daily std by sqrt(252).
# GDX should be persistently more volatile — equity beta layered on top of gold price risk.

roll_vol_gld = returns['ret_GLD'].rolling(30).std() * np.sqrt(252) * 100  # as %
roll_vol_gdx = returns['ret_GDX'].rolling(30).std() * np.sqrt(252) * 100

avg_gld = roll_vol_gld.mean()
avg_gdx = roll_vol_gdx.mean()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(roll_vol_gld.index, roll_vol_gld, color='goldenrod', linewidth=1.3, label='GLD volatility')
ax.plot(roll_vol_gdx.index, roll_vol_gdx, color='steelblue',  linewidth=1.3, label='GDX volatility')
ax.axhline(avg_gld, color='goldenrod', linestyle=':', linewidth=1.0, alpha=0.7,
           label=f'GLD avg: {avg_gld:.1f}%')
ax.axhline(avg_gdx, color='steelblue',  linestyle=':', linewidth=1.0, alpha=0.7,
           label=f'GDX avg: {avg_gdx:.1f}%')

ax.set_title('Rolling 30-Day Annualised Volatility: GLD vs. GDX', fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Annualised Volatility (%)')
ax.legend()
plt.tight_layout()
plt.savefig('data/plot3_rolling_volatility.png', dpi=150)
plt.show()

print(f"""
Caption: GDX is consistently more volatile than GLD across the full period
(avg. {avg_gdx:.1f}% vs. {avg_gld:.1f}% annualised). Volatility clusters are
visible in both series — the 2020 COVID spike and 2022 rate shock are prominent.
The vol ratio GDX/GLD is not constant over time, which is precisely why a static
hedge ratio is insufficient — the Kalman filter in Step 7 will allow the ratio to
adapt dynamically to this changing risk relationship.
""")

In [ ]:
# --- Plot 4: Log Price Ratio (Naive Spread) ---
# log(GLD/GDX) = log_GLD - log_GDX assumes a 1:1 hedge ratio.
# This is almost certainly wrong — but plotting it first makes the motivation
# for estimating a proper hedge ratio visually obvious.

df['log_ratio'] = df['log_GLD'] - df['log_GDX']
ratio_mean = df['log_ratio'].mean()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df.index, df['log_ratio'], color='darkorchid', linewidth=1.2, label='log(GLD) − log(GDX)')
ax.axhline(ratio_mean, color='crimson', linestyle='--', linewidth=1.2,
           label=f'Full-period mean: {ratio_mean:.3f}')

ax.set_title('Log Price Ratio: log(GLD) − log(GDX) — Naive Spread', fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('log(GLD / GDX)')
ax.legend()
plt.tight_layout()
plt.savefig('data/plot4_log_ratio.png', dpi=150)
plt.show()

print("""
Caption: The naive log ratio shows a clear upward trend over the full period —
it is NOT stationary and does not mean-revert around a fixed level. A 1:1 hedge
ratio is therefore invalid as a pairs trade. This motivates the OLS-estimated
hedge ratio (Step 5) and the time-varying Kalman filter hedge ratio (Step 7).
The long-run upward drift reflects the structural de-rating of gold mining equities
relative to physical gold: rising operational costs, poor capital allocation, and
equity market beta all weighed on GDX relative to GLD over this period.
""")

In [ ]:
# --- Plot 5: Return Distributions ---
# KDE + histogram with fitted normal curve. Financial returns are fat-tailed
# (leptokurtic) — extreme moves are more frequent than a Gaussian model predicts.
# This matters when interpreting z-score signal thresholds in Step 8.

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, col, color, label in zip(
    axes,
    ['ret_GLD', 'ret_GDX'],
    ['goldenrod', 'steelblue'],
    ['GLD', 'GDX']
):
    data = returns[col].dropna()
    mu, sigma = data.mean(), data.std()
    x_range = np.linspace(data.min(), data.max(), 300)

    ax.hist(data, bins=80, density=True, alpha=0.4, color=color, label='Empirical')
    ax.plot(x_range, stats.gaussian_kde(data)(x_range), color=color, linewidth=2, label='KDE')
    ax.plot(x_range, stats.norm.pdf(x_range, mu, sigma),
            color='crimson', linestyle='--', linewidth=1.5,
            label=f'Normal fit (σ={sigma:.4f})')

    kurt = data.kurt()
    skew = data.skew()
    ax.text(0.97, 0.95, f'Excess Kurt: {kurt:.2f}\nSkew: {skew:.2f}',
            transform=ax.transAxes, ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    ax.set_title(f'{label} Daily Log Return Distribution', fontweight='bold')
    ax.set_xlabel('Log Return')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('data/plot5_return_distributions.png', dpi=150)
plt.show()

gld_kurt = returns['ret_GLD'].kurt()
gdx_kurt = returns['ret_GDX'].kurt()
print(f"""
Caption: Both GLD and GDX returns show pronounced fat tails relative to a normal
distribution — excess kurtosis of {gld_kurt:.2f} (GLD) and {gdx_kurt:.2f} (GDX).
A normal distribution has excess kurtosis of 0; values well above zero confirm that
large daily moves occur far more often than a Gaussian model would predict. This has
a direct implication for the z-score signal thresholds set in Step 8: ±2 does not
correspond to a clean 95th percentile in practice — extreme spread observations will
fire more frequently than expected under normality.
""")

---
## Step 4: Stationarity Testing (ADF)

Before testing for cointegration, we must confirm that both log price series are **integrated of order 1** — I(1). This means:
- The raw log price series are **non-stationary** (ADF fails to reject a unit root)
- The first differences (log returns) are **stationary** (ADF rejects a unit root)

This is a prerequisite for cointegration: two I(1) series can share a long-run equilibrium that is itself stationary (I(0)). If one series were I(0) or I(2), cointegration as defined by Engle-Granger would not apply.

We use the **Augmented Dickey-Fuller (ADF)** test with `autolag='AIC'` to automatically select the number of lags. The null hypothesis is that a unit root is present (non-stationary). We reject the null at the 5% level when p-value < 0.05.

In [ ]:
def run_adf(series: pd.Series, name: str) -> dict:
    """
    Run ADF test and return key statistics as a dict.

    Args:
        series: time series to test
        name:   label for display

    Returns:
        dict with test stat, p-value, lags used, critical values, and conclusion
    """
    result = adfuller(series.dropna(), autolag='AIC')
    stat, pval, lags, _, crit, _ = result[0], result[1], result[2], result[3], result[4], result[5]
    conclusion = 'Stationary (reject H₀)' if pval < 0.05 else 'Non-stationary (fail to reject H₀)'
    return {
        'Series'       : name,
        'ADF Statistic': round(stat, 4),
        'p-value'      : round(pval, 4),
        'Lags Used'    : lags,
        'Crit 1%'      : round(crit['1%'], 4),
        'Crit 5%'      : round(crit['5%'], 4),
        'Crit 10%'     : round(crit['10%'], 4),
        'Conclusion'   : conclusion,
    }


# Test raw log prices (expect non-stationary)
adf_log_gld = run_adf(df['log_GLD'],        'log(GLD)')
adf_log_gdx = run_adf(df['log_GDX'],        'log(GDX)')

# Test first differences / log returns (expect stationary)
adf_ret_gld = run_adf(returns['ret_GLD'],   'Δlog(GLD)')
adf_ret_gdx = run_adf(returns['ret_GDX'],   'Δlog(GDX)')

adf_results = pd.DataFrame([adf_log_gld, adf_log_gdx, adf_ret_gld, adf_ret_gdx])
adf_results = adf_results.set_index('Series')

print("=== ADF Stationarity Test Results ===\n")
display(adf_results)

In [ ]:
# Visual confirmation: plot log prices alongside their first differences
fig, axes = plt.subplots(2, 2, figsize=(13, 7))

axes[0, 0].plot(df['log_GLD'], color='goldenrod', linewidth=1.2)
axes[0, 0].set_title('log(GLD) — Level', fontweight='bold')
axes[0, 0].set_ylabel('Log Price')

axes[0, 1].plot(df['log_GDX'], color='steelblue', linewidth=1.2)
axes[0, 1].set_title('log(GDX) — Level', fontweight='bold')
axes[0, 1].set_ylabel('Log Price')

axes[1, 0].plot(returns['ret_GLD'], color='goldenrod', linewidth=0.8, alpha=0.8)
axes[1, 0].axhline(0, color='black', linewidth=0.7)
axes[1, 0].set_title('Δlog(GLD) — First Difference', fontweight='bold')
axes[1, 0].set_ylabel('Log Return')

axes[1, 1].plot(returns['ret_GDX'], color='steelblue', linewidth=0.8, alpha=0.8)
axes[1, 1].axhline(0, color='black', linewidth=0.7)
axes[1, 1].set_title('Δlog(GDX) — First Difference', fontweight='bold')
axes[1, 1].set_ylabel('Log Return')

for ax in axes.flat:
    ax.set_xlabel('Date')

plt.suptitle('Log Prices (Non-Stationary) vs. Log Returns (Stationary)', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('data/plot_adf_visual.png', dpi=150)
plt.show()

print("""
Interpretation: The top row (log prices) shows clear trending behaviour with no fixed
mean — consistent with a unit root (non-stationary). The bottom row (log returns /
first differences) fluctuates around zero with no trend — consistent with stationarity.
Both series are confirmed I(1): the prerequisite for cointegration testing is satisfied.
""")

---
## Step 5: Cointegration Testing

Having established that both series are I(1), we now test whether a linear combination of them is I(0) — i.e., stationary. If so, the pair is **cointegrated**: they share a long-run equilibrium and any deviation from it is temporary and mean-reverting. This is the formal foundation for a pairs trade.

We run two independent tests:

1. **Engle-Granger (1987):** OLS regression of `log(GLD)` on `log(GDX)` and a constant, then ADF test on the residuals. If residuals are stationary, cointegration is confirmed. The OLS coefficient is the **static hedge ratio β**.

2. **Johansen (1988):** A VAR-based multivariate approach that tests for the number of cointegrating relationships. More powerful than Engle-Granger and doesn't require choosing a dependent variable. Both the trace statistic and max-eigenvalue statistic are reported.

If both tests confirm cointegration, we have strong evidence for a stable long-run relationship.

In [ ]:
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

# --- Engle-Granger Test ---
# Regress log(GLD) on log(GDX) + constant.
# The cointegrating vector is extracted from the OLS coefficients.

X = add_constant(df['log_GDX'])   # design matrix: [intercept, log_GDX]
Y = df['log_GLD']

ols_model  = OLS(Y, X).fit()
eg_alpha   = ols_model.params['const']          # intercept
eg_beta    = ols_model.params['log_GDX']        # static hedge ratio
eg_resid   = ols_model.resid                    # spread residuals
eg_r2      = ols_model.rsquared

print("=== Engle-Granger Step 1: OLS Regression ===")
print(f"  Equation : log(GLD) = {eg_alpha:.4f} + {eg_beta:.4f} * log(GDX)")
print(f"  R²       : {eg_r2:.4f}")
print(f"  Intercept (α) : {eg_alpha:.4f}")
print(f"  Hedge ratio (β): {eg_beta:.4f}")
print()

# ADF on residuals — using no constant/trend since residuals are mean-zero by construction
eg_adf = adfuller(eg_resid, autolag='AIC', regression='n')
eg_stat, eg_pval, eg_lags, _, eg_crit, _ = eg_adf

print("=== Engle-Granger Step 2: ADF on OLS Residuals ===")
print(f"  ADF Statistic : {eg_stat:.4f}")
print(f"  p-value       : {eg_pval:.4f}")
print(f"  Lags Used     : {eg_lags}")
print(f"  Critical 1%   : {eg_crit['1%']:.4f}")
print(f"  Critical 5%   : {eg_crit['5%']:.4f}")
print(f"  Critical 10%  : {eg_crit['10%']:.4f}")
eg_conclusion = 'COINTEGRATED ✓' if eg_pval < 0.05 else 'NOT cointegrated ✗'
print(f"\n  Conclusion: {eg_conclusion} (p={'<0.0001' if eg_pval < 0.0001 else round(eg_pval,4)})")

In [ ]:
from statsmodels.tsa.vector_ar.vecm import coint_johansen

# --- Johansen Test ---
# det_order=0: include a constant in the cointegrating relation (appropriate here)
# k_ar_diff=1: one lag in the VAR in first differences (standard starting point)

joh = coint_johansen(df[['log_GLD', 'log_GDX']], det_order=0, k_ar_diff=1)

# Trace statistic: H0 = at most r cointegrating relations
# Max-eigenvalue:  H0 = exactly r cointegrating relations
# Critical values at index 0=90%, 1=95%, 2=99%

print("=== Johansen Cointegration Test ===\n")
print("Trace Statistic (H₀: r ≤ k):")
trace_header = f"  {'r':>4}  {'Trace Stat':>12}  {'Crit 90%':>10}  {'Crit 95%':>10}  {'Crit 99%':>10}  {'Reject H₀ at 5%':>16}"
print(trace_header)
print("  " + "-" * (len(trace_header) - 2))
for i in range(len(joh.lr1)):
    stat  = joh.lr1[i]
    c90, c95, c99 = joh.cvt[i]
    reject = "Yes ✓" if stat > c95 else "No ✗"
    print(f"  {i:>4}  {stat:>12.4f}  {c90:>10.4f}  {c95:>10.4f}  {c99:>10.4f}  {reject:>16}")

print()
print("Max-Eigenvalue Statistic (H₀: r = k):")
maxeig_header = f"  {'r':>4}  {'MaxEig Stat':>12}  {'Crit 90%':>10}  {'Crit 95%':>10}  {'Crit 99%':>10}  {'Reject H₀ at 5%':>16}"
print(maxeig_header)
print("  " + "-" * (len(maxeig_header) - 2))
for i in range(len(joh.lr2)):
    stat  = joh.lr2[i]
    c90, c95, c99 = joh.cvm[i]
    reject = "Yes ✓" if stat > c95 else "No ✗"
    print(f"  {i:>4}  {stat:>12.4f}  {c90:>10.4f}  {c95:>10.4f}  {c99:>10.4f}  {reject:>16}")

print()
print("Eigenvectors (cointegrating vectors — columns):")
print(joh.evec)
print()

# Extract Johansen hedge ratio from the first eigenvector (normalised to GLD=1)
joh_vec   = joh.evec[:, 0]
joh_beta  = -joh_vec[1] / joh_vec[0]   # normalise so GLD coefficient = 1
print(f"Johansen implied hedge ratio (β): {joh_beta:.4f}")

In [ ]:
# --- Summary ---
print("=== Cointegration Test Summary ===\n")
print(f"  Engle-Granger ADF on residuals : p = {round(eg_pval, 4)}  →  {eg_conclusion}")
joh_trace_reject = joh.lr1[0] > joh.cvt[0][1]
joh_conclusion   = 'COINTEGRATED ✓' if joh_trace_reject else 'NOT cointegrated over full period ✗'
print(f"  Johansen trace test (r=0)      : stat = {joh.lr1[0]:.4f}, 5% crit = {joh.cvt[0][1]:.4f}  →  {joh_conclusion}")
print(f"\n  OLS R²           : {eg_r2:.4f}")
print(f"  OLS β (hedge)    : {eg_beta:.4f}")
print(f"  OLS α (intercept): {eg_alpha:.4f}")

print("""
Interpretation: Both tests fail to confirm cointegration over the full 2010–2024 period.
This is a meaningful economic finding, not a failure of the analysis.

GLD approximately doubled in price while GDX declined from ~$55 to ~$34 — these series
moved in structurally opposite directions over 14 years. The OLS R² of 0.35 and the
upward-trending log ratio (Plot 4) already hinted at this. A permanent divergence of
this magnitude is not a mean-reverting spread; it reflects a structural de-rating of
gold mining equities driven by cost inflation, capex destruction, and persistent
equity risk premium expansion in the mining sector.

This motivates the key question: was there a period where GLD/GDX WAS cointegrated,
and when did that relationship break down? Rolling cointegration analysis below
addresses this directly.
""")

# Store OLS parameters — we still use them for sub-period spread construction
STATIC_BETA  = eg_beta
STATIC_ALPHA = eg_alpha

### Step 5b: Rolling Cointegration Analysis

Since full-period cointegration does not hold, we test it on a rolling 2-year (504 trading day) window. For each window, we run the Engle-Granger ADF test on OLS residuals and record the p-value. This tells us *when* the cointegration relationship was active and when it broke down — which is the analytically interesting question.

In [ ]:
def rolling_eg_pvalue(log_y: pd.Series, log_x: pd.Series, window: int = 504) -> pd.Series:
    """
    Compute rolling Engle-Granger cointegration p-value over a sliding window.

    For each window: OLS regress log_y on log_x, run ADF on residuals,
    return the ADF p-value at the end date of each window.
    """
    pvals = {}
    for end in range(window, len(log_y)):
        y = log_y.iloc[end - window: end]
        x = log_x.iloc[end - window: end]
        resid = OLS(y, add_constant(x)).fit().resid
        pval  = adfuller(resid, autolag='AIC', regression='n')[1]
        pvals[log_y.index[end]] = pval
    return pd.Series(pvals)


print("Computing rolling cointegration p-values (2-year window)... this may take ~30s")
roll_coint_pval = rolling_eg_pvalue(df['log_GLD'], df['log_GDX'], window=504)
print("Done.")

# Also compute rolling OLS beta over the same window
roll_beta = {}
for end in range(504, len(df)):
    y = df['log_GLD'].iloc[end - 504: end]
    x = df['log_GDX'].iloc[end - 504: end]
    b = OLS(y, add_constant(x)).fit().params['log_GDX']
    roll_beta[df.index[end]] = b
roll_beta = pd.Series(roll_beta)

# Plot rolling p-value + rolling beta side by side
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# Panel 1: p-value
axes[0].plot(roll_coint_pval.index, roll_coint_pval, color='darkorchid', linewidth=1.2,
             label='Rolling EG p-value (2yr window)')
axes[0].axhline(0.05, color='crimson', linestyle='--', linewidth=1.2, label='5% significance threshold')
axes[0].fill_between(roll_coint_pval.index, roll_coint_pval, 0.05,
                     where=(roll_coint_pval > 0.05), alpha=0.15, color='crimson',
                     label='Non-cointegrated periods')
axes[0].set_ylabel('ADF p-value on residuals')
axes[0].set_title('Rolling 2-Year Cointegration Strength: GLD/GDX', fontweight='bold')
axes[0].set_ylim(0, 0.6)
axes[0].legend(fontsize=9)

# Panel 2: rolling hedge ratio
axes[1].plot(roll_beta.index, roll_beta, color='steelblue', linewidth=1.2,
             label='Rolling OLS β (hedge ratio)')
axes[1].axhline(STATIC_BETA, color='grey', linestyle='--', linewidth=1.0,
                label=f'Full-period OLS β = {STATIC_BETA:.3f}')
axes[1].set_ylabel('Hedge Ratio β')
axes[1].set_xlabel('Date')
axes[1].set_title('Rolling 2-Year OLS Hedge Ratio', fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('data/plot_rolling_cointegration.png', dpi=150)
plt.show()

# Summarise periods of cointegration
coint_periods = roll_coint_pval[roll_coint_pval < 0.05]
non_coint_periods = roll_coint_pval[roll_coint_pval >= 0.05]
pct_coint = len(coint_periods) / len(roll_coint_pval) * 100
print(f"\nWindows tested        : {len(roll_coint_pval)}")
print(f"Cointegrated (p<0.05) : {len(coint_periods)} ({pct_coint:.1f}% of windows)")
print(f"Non-cointegrated      : {len(non_coint_periods)} ({100-pct_coint:.1f}% of windows)")
print(f"\nCointegration was strongest (lowest p-value):")
print(f"  Date: {roll_coint_pval.idxmin().date()}  p = {roll_coint_pval.min():.4f}")
print(f"\nRolling β range: {roll_beta.min():.3f} to {roll_beta.max():.3f}")
print(f"  (wide range confirms hedge ratio is not stable — motivates Kalman filter in Step 7)")